Import Libraries and Select the Solar System

In [1]:
import requests
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd

BASE = "https://oedi-data-lake.s3.amazonaws.com/"
SYSTEM_ID = 1283

# This tells Python how to read the cloud's folder listing.
NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}

def list_folder(prefix):
    response = requests.get(
        BASE,
        params={
            "list-type": "2",
            "prefix": prefix,
            "delimiter": "/"
        },
        timeout=30
    )
    response.raise_for_status()
    return ET.fromstring(response.content)

Discover Available Years and Months

In [2]:
system_folder = (
    f"pvdaq/csv/pvdata/system_id={SYSTEM_ID}/"
)

listing = list_folder(system_folder)

year_folders = [
    item.text
    for item in listing.findall(
        "s3:CommonPrefixes/s3:Prefix", NS
    )
]

year_folders = sorted(
    year_folders,
    key=lambda folder: int(folder.split("year=")[1].split("/")[0])
)

print("Available year folders:")
for folder in year_folders:
    print(folder)

if not year_folders:
    raise ValueError("No CSV year folders found for this system.")

latest_year_folder = year_folders[-1]
print("\nNewest available year folder:", latest_year_folder)

Available year folders:
pvdaq/csv/pvdata/system_id=1283/year=2012/
pvdaq/csv/pvdata/system_id=1283/year=2013/
pvdaq/csv/pvdata/system_id=1283/year=2014/
pvdaq/csv/pvdata/system_id=1283/year=2015/
pvdaq/csv/pvdata/system_id=1283/year=2016/
pvdaq/csv/pvdata/system_id=1283/year=2017/
pvdaq/csv/pvdata/system_id=1283/year=2018/
pvdaq/csv/pvdata/system_id=1283/year=2019/
pvdaq/csv/pvdata/system_id=1283/year=2020/
pvdaq/csv/pvdata/system_id=1283/year=2022/

Newest available year folder: pvdaq/csv/pvdata/system_id=1283/year=2022/


In [3]:
listing = list_folder(latest_year_folder)

month_folders = [
    item.text
    for item in listing.findall(
        "s3:CommonPrefixes/s3:Prefix", NS
    )
]

month_folders = sorted(
    month_folders,
    key=lambda folder: int(folder.split("month=")[1].split("/")[0])
)

print("Available month folders:")
for folder in month_folders:
    print(folder)

if not month_folders:
    raise ValueError("No month folders found.")

selected_month_folder = month_folders[-1]
print("\nMonth selected:", selected_month_folder)

Available month folders:
pvdaq/csv/pvdata/system_id=1283/year=2022/month=2/

Month selected: pvdaq/csv/pvdata/system_id=1283/year=2022/month=2/


In [4]:
raw_folder = Path("solar_data/data/raw")
raw_folder.mkdir(parents=True, exist_ok=True)

listing = list_folder(selected_month_folder)

day_folders = [
    item.text
    for item in listing.findall(
        "s3:CommonPrefixes/s3:Prefix", NS
    )
]

download_log = []

for day_folder in day_folders:
    day_listing = list_folder(day_folder)

    for item in day_listing.findall("s3:Contents", NS):
        key = item.find("s3:Key", NS).text

        if not key.endswith(".csv"):
            continue

        url = BASE + key
        filename = key.split("/")[-1]

        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()

            # Save the original bytes without changing the file.
            (raw_folder / filename).write_bytes(response.content)
            status = "downloaded"

        except requests.RequestException as error:
            status = str(error)

        download_log.append({
            "filename": filename,
            "url": url,
            "status": status
        })

log = pd.DataFrame(download_log)
log.to_csv("solar_data/download_log.csv", index=False)

display(log)
print("CSV files saved:", len(list(raw_folder.glob("*.csv"))))

,filename,url,status
0,system_1283__date_2022_02_01.csv,https://oedi-data-lake.s3.amazonaws.com/pvdaq/...,downloaded
1,system_1283__date_2022_02_02.csv,https://oedi-data-lake.s3.amazonaws.com/pvdaq/...,downloaded
2,system_1283__date_2022_02_03.csv,https://oedi-data-lake.s3.amazonaws.com/pvdaq/...,downloaded
3,system_1283__date_2022_02_04.csv,https://oedi-data-lake.s3.amazonaws.com/pvdaq/...,downloaded
4,system_1283__date_2022_02_05.csv,https://oedi-data-lake.s3.amazonaws.com/pvdaq/...,downloaded
5,system_1283__date_2022_02_06.csv,https://oedi-data-lake.s3.amazonaws.com/pvdaq/...,downloaded
6,system_1283__date_2022_02_07.csv,https://oedi-data-lake.s3.amazonaws.com/pvdaq/...,downloaded
7,system_1283__date_2022_02_08.csv,https://oedi-data-lake.s3.amazonaws.com/pvdaq/...,downloaded


CSV files saved: 8


Download and Inspect an Initial Sample

In [5]:
downloaded_files = sorted(raw_folder.glob("*.csv"))

if not downloaded_files:
    raise ValueError("No CSV files downloaded. Check the previous output.")

sample = pd.read_csv(downloaded_files[0])

print("File:", downloaded_files[0].name)
print("Rows and columns:", sample.shape)
print("\nColumn names:")
print(sample.columns.tolist())

display(sample.head())

File: system_1283__date_2022_02_01.csv
Rows and columns: (5760, 22)

Column names:
['measured_on', 'ac_meter_1_power_kw__1041', 'ac_meter_2_power_kw__1042', 'ac_power__1137', 'ac_power_metered_kw__1040', 'ambient_temp__1053', 'inv1_ac_power_kw__1043', 'inv1_dc_current__1045', 'inv1_dc_power__1134', 'inv1_dc_voltage__1044', 'inv1_temp__1046', 'inv2_ac_power_kw__1047', 'inv2_dc_current__1049', 'inv2_dc_power__1135', 'inv2_dc_voltage__1048', 'inv2_temp__1050', 'module_temp__1056', 'poa_irradiance__1055', 'poa_irradiance_refcell__1054', 'refcell_temp__1052', 'wind_speed__1051', 'system_id']


,measured_on,ac_meter_1_power_kw__1041,ac_meter_2_power_kw__1042,ac_power__1137,ac_power_metered_kw__1040,ambient_temp__1053,inv1_ac_power_kw__1043,inv1_dc_current__1045,inv1_dc_power__1134,inv1_dc_voltage__1044,...,inv2_dc_current__1049,inv2_dc_power__1135,inv2_dc_voltage__1048,inv2_temp__1050,module_temp__1056,poa_irradiance__1055,poa_irradiance_refcell__1054,refcell_temp__1052,wind_speed__1051,system_id
0,2022-02-01 00:00:00,0.0,0.0,0.0,0.0,3.095306,0.0,0.0,0.0,0.0,...,0.0,0.0,3.600098,28.40015,-6.387450,0.0,-1.831055,-0.583333,3.430851,1283
1,2022-02-01 00:00:15,0.0,0.0,0.0,0.0,3.090975,0.0,0.0,0.0,0.0,...,0.0,0.0,3.600098,28.34676,-6.414577,0.0,-1.831055,-0.621989,3.440351,1283
2,2022-02-01 00:00:30,0.0,0.0,0.0,0.0,3.076898,0.0,0.0,0.0,0.0,...,0.0,0.0,3.600098,28.35344,-6.431191,0.0,-1.871680,-0.674886,3.418500,1283
3,2022-02-01 00:00:45,0.0,0.0,0.0,0.0,3.073108,0.0,0.0,0.0,0.0,...,0.0,0.0,3.600098,28.34676,-6.467473,0.0,-1.942773,-0.701335,3.411350,1283
4,2022-02-01 00:01:00,0.0,0.0,0.0,0.0,3.076031,0.0,0.0,0.0,0.0,...,0.0,0.0,3.600098,28.30005,-6.497652,0.0,-1.831055,-0.720662,3.408350,1283


Set Up AWS Access and Download Historical Data

In [6]:
!pip -q install awscli

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 30.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


In [7]:
!aws s3 ls "s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/" --no-sign-request

                           PRE year=2012/
                           PRE year=2013/
                           PRE year=2014/
                           PRE year=2015/
                           PRE year=2016/
                           PRE year=2017/
                           PRE year=2018/
                           PRE year=2019/
                           PRE year=2020/
                           PRE year=2022/


In [8]:
!aws s3 cp "s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2014/month=1/" "./solar_data/data/raw/system_1283/year_2014/month_01/" --recursive --exclude "*" --include "*.csv" --no-sign-request

download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2014/month=1/day=18/system_1283__date_2014_01_18.csv to solar_data/data/raw/system_1283/year_2014/month_01/day=18/system_1283__date_2014_01_18.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2014/month=1/day=11/system_1283__date_2014_01_11.csv to solar_data/data/raw/system_1283/year_2014/month_01/day=11/system_1283__date_2014_01_11.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2014/month=1/day=15/system_1283__date_2014_01_15.csv to solar_data/data/raw/system_1283/year_2014/month_01/day=15/system_1283__date_2014_01_15.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2014/month=1/day=17/system_1283__date_2014_01_17.csv to solar_data/data/raw/system_1283/year_2014/month_01/day=17/system_1283__date_2014_01_17.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2014/month=1/day=16/system_1283__date_2014_01_16.csv to solar_data/data/raw/s

In [9]:
!aws s3 cp "s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2014/" "./solar_data/data/raw/system_1283/year_2014/" --recursive --exclude "*" --include "*.csv" --no-sign-request

download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2014/month=1/day=1/system_1283__date_2014_01_01.csv to solar_data/data/raw/system_1283/year_2014/month=1/day=1/system_1283__date_2014_01_01.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2014/month=1/day=14/system_1283__date_2014_01_14.csv to solar_data/data/raw/system_1283/year_2014/month=1/day=14/system_1283__date_2014_01_14.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2014/month=1/day=15/system_1283__date_2014_01_15.csv to solar_data/data/raw/system_1283/year_2014/month=1/day=15/system_1283__date_2014_01_15.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2014/month=1/day=16/system_1283__date_2014_01_16.csv to solar_data/data/raw/system_1283/year_2014/month=1/day=16/system_1283__date_2014_01_16.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2014/month=1/day=12/system_1283__date_2014_01_12.csv to solar_data/data/raw/system_

In [10]:
!aws s3 cp "s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2015/" "./solar_data/data/raw/system_1283/year_2015/" --recursive --exclude "*" --include "*.csv" --no-sign-request

download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2015/month=1/day=1/system_1283__date_2015_01_01.csv to solar_data/data/raw/system_1283/year_2015/month=1/day=1/system_1283__date_2015_01_01.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2015/month=1/day=2/system_1283__date_2015_01_02.csv to solar_data/data/raw/system_1283/year_2015/month=1/day=2/system_1283__date_2015_01_02.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2015/month=1/day=11/system_1283__date_2015_01_11.csv to solar_data/data/raw/system_1283/year_2015/month=1/day=11/system_1283__date_2015_01_11.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2015/month=1/day=13/system_1283__date_2015_01_13.csv to solar_data/data/raw/system_1283/year_2015/month=1/day=13/system_1283__date_2015_01_13.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2015/month=1/day=15/system_1283__date_2015_01_15.csv to solar_data/data/raw/system_12

In [11]:
import shutil
from google.colab import files

shutil.make_archive("solar_raw_data", "zip", "solar_data/data/raw")
files.download("solar_raw_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Preserve and Export Raw Data

In [12]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/solar_1283_2014_2015",
    "zip",
    "/content/solar_data/data/raw/system_1283"
)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Check quality of data

In [13]:
from pathlib import Path
import pandas as pd

folder = Path("/content/solar_data/data/raw/system_1283")

required = [
    "ac_power_metered_kw__1040",
    "poa_irradiance__1055",
    "ambient_temp__1053"
]

# Select only the 2014 and 2015 files.
csv_files = sorted(
    file for file in folder.rglob("*.csv")
    if "__date_2014_" in file.name or "__date_2015_" in file.name
)

if not csv_files:
    raise ValueError("No 2014–2015 CSV files found. Check the folder path.")

results = []

for file in csv_files:
    file_date = pd.to_datetime(
        file.stem.split("__date_")[-1],
        format="%Y_%m_%d"
    )

    row = {
        "date": file_date,
        "file": file.name
    }

    try:
        df = pd.read_csv(file)
        row["rows"] = len(df)
        row["exact_duplicate_rows"] = df.duplicated().sum()

        missing_columns = [
            col for col in ["measured_on"] + required
            if col not in df.columns
        ]
        row["missing_columns"] = ", ".join(missing_columns)

        if "measured_on" in df.columns:
            timestamps = pd.to_datetime(
                df["measured_on"], errors="coerce"
            )
            valid_times = timestamps.dropna()

            row["invalid_timestamps"] = timestamps.isna().sum()
            row["duplicate_timestamps"] = valid_times.duplicated().sum()
            row["first_time"] = valid_times.min()
            row["last_time"] = valid_times.max()

            # Use unique, sorted timestamps to examine spacing.
            intervals = (
                valid_times.drop_duplicates()
                .sort_values()
                .diff()
                .dt.total_seconds()
                .dropna()
            )

            typical_interval = intervals.median()
            row["median_interval_seconds"] = typical_interval
            row["largest_gap_seconds"] = intervals.max()

            # Diagnostic only—not an automatic rejection rule.
            row["gaps_above_1_5x_median"] = (
                intervals > 1.5 * typical_interval
            ).sum()

            row["timestamps_on_other_dates"] = (
                valid_times.dt.normalize() != file_date
            ).sum()

        for col in required:
            if col in df.columns:
                values = pd.to_numeric(df[col], errors="coerce")

                row[f"{col}_missing_pct"] = (
                    values.isna().mean() * 100
                )
                row[f"{col}_nonnumeric_count"] = (
                    df[col].notna() & values.isna()
                ).sum()
                row[f"{col}_min"] = values.min()
                row[f"{col}_max"] = values.max()

        row["read_error"] = ""

    except Exception as error:
        row["read_error"] = str(error)

    results.append(row)

quality = pd.DataFrame(results).sort_values("date")
quality.to_csv("/content/solar_quality_report.csv", index=False)

display(quality.head())
print("Files inspected:", len(quality))
print("Files that could not be read:",
      quality["read_error"].ne("").sum())

,date,file,rows,exact_duplicate_rows,missing_columns,invalid_timestamps,duplicate_timestamps,first_time,last_time,median_interval_seconds,...,ac_power_metered_kw__1040_max,poa_irradiance__1055_missing_pct,poa_irradiance__1055_nonnumeric_count,poa_irradiance__1055_min,poa_irradiance__1055_max,ambient_temp__1053_missing_pct,ambient_temp__1053_nonnumeric_count,ambient_temp__1053_min,ambient_temp__1053_max,read_error
0,2014-01-01,system_1283__date_2014_01_01.csv,5760,0,,0,0,2014-01-01,2014-01-01 23:59:45,15.0,...,302.8,0.0,0,0.0,603.5763,0.0,0,-5.138009,7.40472,
351,2014-01-01,system_1283__date_2014_01_01.csv,5760,0,,0,0,2014-01-01,2014-01-01 23:59:45,15.0,...,302.8,0.0,0,0.0,603.5763,0.0,0,-5.138009,7.40472,
362,2014-01-02,system_1283__date_2014_01_02.csv,5760,0,,0,0,2014-01-02,2014-01-02 23:59:45,15.0,...,282.1,0.0,0,0.0,568.7867,0.0,0,-5.774758,10.10927,
11,2014-01-02,system_1283__date_2014_01_02.csv,5760,0,,0,0,2014-01-02,2014-01-02 23:59:45,15.0,...,282.1,0.0,0,0.0,568.7867,0.0,0,-5.774758,10.10927,
373,2014-01-03,system_1283__date_2014_01_03.csv,5760,0,,0,0,2014-01-03,2014-01-03 23:59:45,15.0,...,257.2,0.0,0,0.0,573.0586,0.0,0,2.071747,17.67985,


Files inspected: 744
Files that could not be read: 0


In [14]:
display(quality.head(20))

,date,file,rows,exact_duplicate_rows,missing_columns,invalid_timestamps,duplicate_timestamps,first_time,last_time,median_interval_seconds,...,ac_power_metered_kw__1040_max,poa_irradiance__1055_missing_pct,poa_irradiance__1055_nonnumeric_count,poa_irradiance__1055_min,poa_irradiance__1055_max,ambient_temp__1053_missing_pct,ambient_temp__1053_nonnumeric_count,ambient_temp__1053_min,ambient_temp__1053_max,read_error
0,2014-01-01,system_1283__date_2014_01_01.csv,5760,0,,0,0,2014-01-01,2014-01-01 23:59:45,15.0,...,302.8,0.0,0,0.0,603.5763,0.0,0,-5.138009,7.404720,
351,2014-01-01,system_1283__date_2014_01_01.csv,5760,0,,0,0,2014-01-01,2014-01-01 23:59:45,15.0,...,302.8,0.0,0,0.0,603.5763,0.0,0,-5.138009,7.404720,
362,2014-01-02,system_1283__date_2014_01_02.csv,5760,0,,0,0,2014-01-02,2014-01-02 23:59:45,15.0,...,282.1,0.0,0,0.0,568.7867,0.0,0,-5.774758,10.109270,
11,2014-01-02,system_1283__date_2014_01_02.csv,5760,0,,0,0,2014-01-02,2014-01-02 23:59:45,15.0,...,282.1,0.0,0,0.0,568.7867,0.0,0,-5.774758,10.109270,
373,2014-01-03,system_1283__date_2014_01_03.csv,5760,0,,0,0,2014-01-03,2014-01-03 23:59:45,15.0,...,257.2,0.0,0,0.0,573.0586,0.0,0,2.071747,17.679850,
22,2014-01-03,system_1283__date_2014_01_03.csv,5760,0,,0,0,2014-01-03,2014-01-03 23:59:45,15.0,...,257.2,0.0,0,0.0,573.0586,0.0,0,2.071747,17.679850,
376,2014-01-04,system_1283__date_2014_01_04.csv,5756,0,,0,0,2014-01-04,2014-01-04 23:59:45,15.0,...,0.2,0.0,0,0.0,197.5096,0.0,0,-12.578420,4.600541,
25,2014-01-04,system_1283__date_2014_01_04.csv,5756,0,,0,0,2014-01-04,2014-01-04 23:59:45,15.0,...,0.2,0.0,0,0.0,197.5096,0.0,0,-12.578420,4.600541,
26,2014-01-05,system_1283__date_2014_01_05.csv,5760,0,,0,0,2014-01-05,2014-01-05 23:59:45,15.0,...,22.3,0.0,0,0.0,783.6914,0.0,0,-17.372440,-7.707524,
377,2014-01-05,system_1283__date_2014_01_05.csv,5760,0,,0,0,2014-01-05,2014-01-05 23:59:45,15.0,...,22.3,0.0,0,0.0,783.6914,0.0,0,-17.372440,-7.707524,


In [15]:
expected_dates = pd.date_range("2014-01-01", "2015-12-31")
available_dates = pd.DatetimeIndex(quality["date"].unique())

missing_dates = expected_dates.difference(available_dates)

print("Expected calendar days:", len(expected_dates))
print("Dates with files:", len(available_dates))
print("Dates without files:", len(missing_dates))

display(pd.DataFrame({"missing_date": missing_dates}))

Expected calendar days: 730
Dates with files: 713
Dates without files: 17


,missing_date
0,2014-07-20
1,2014-07-21
2,2014-07-22
3,2014-07-23
4,2014-07-24
5,2014-07-25
6,2014-07-26
7,2014-07-27
8,2014-07-28
9,2014-07-29


In [16]:
from google.colab import files
files.download("/content/solar_quality_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Identify problem in 2015 data. Now compare 2013 data

In [17]:
!aws s3 cp "s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2013/" "/content/solar_data/data/raw/system_1283/year_2013/" --recursive --exclude "*" --include "*.csv" --no-sign-request

download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2013/month=1/day=10/system_1283__date_2013_01_10.csv to solar_data/data/raw/system_1283/year_2013/month=1/day=10/system_1283__date_2013_01_10.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2013/month=1/day=17/system_1283__date_2013_01_17.csv to solar_data/data/raw/system_1283/year_2013/month=1/day=17/system_1283__date_2013_01_17.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2013/month=1/day=12/system_1283__date_2013_01_12.csv to solar_data/data/raw/system_1283/year_2013/month=1/day=12/system_1283__date_2013_01_12.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2013/month=1/day=18/system_1283__date_2013_01_18.csv to solar_data/data/raw/system_1283/year_2013/month=1/day=18/system_1283__date_2013_01_18.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2013/month=1/day=2/system_1283__date_2013_01_02.csv to solar_data/data/raw/system

Investigate an Alternative Period: 2013–2014

In [18]:
csv_files = sorted(
    file for file in folder.rglob("*.csv")
    if "__date_2013_" in file.name
    or "__date_2014_" in file.name
)

In [19]:
quality.to_csv(
    "/content/solar_quality_report_2013_2014.csv",
    index=False
)

In [20]:
row[f"{col}_minus7999_count"] = (values == -7999).sum()

Troubleshooting and Duplicate-File Investigation

In [21]:
from collections import defaultdict

paths_by_name = defaultdict(list)

for file in csv_files:
    paths_by_name[file.name].append(str(file))

for name, paths in paths_by_name.items():
    if len(paths) > 1:
        print(name)
        for path in paths:
            print(" ", path)

system_1283__date_2014_01_01.csv
  /content/solar_data/data/raw/system_1283/year_2014/month=1/day=1/system_1283__date_2014_01_01.csv
  /content/solar_data/data/raw/system_1283/year_2014/month_01/day=1/system_1283__date_2014_01_01.csv
system_1283__date_2014_01_10.csv
  /content/solar_data/data/raw/system_1283/year_2014/month=1/day=10/system_1283__date_2014_01_10.csv
  /content/solar_data/data/raw/system_1283/year_2014/month_01/day=10/system_1283__date_2014_01_10.csv
system_1283__date_2014_01_11.csv
  /content/solar_data/data/raw/system_1283/year_2014/month=1/day=11/system_1283__date_2014_01_11.csv
  /content/solar_data/data/raw/system_1283/year_2014/month_01/day=11/system_1283__date_2014_01_11.csv
system_1283__date_2014_01_12.csv
  /content/solar_data/data/raw/system_1283/year_2014/month=1/day=12/system_1283__date_2014_01_12.csv
  /content/solar_data/data/raw/system_1283/year_2014/month_01/day=12/system_1283__date_2014_01_12.csv
system_1283__date_2014_01_13.csv
  /content/solar_data/dat

In [22]:
from google.colab import files
files.download("/content/solar_quality_report_2013_2014.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
csv_files = sorted(
    file for file in folder.rglob("*.csv")
    if "__date_2013_" in file.name
    or "__date_2014_" in file.name
)

print("2013 files:", sum("__date_2013_" in f.name for f in csv_files))
print("2014 files:", sum("__date_2014_" in f.name for f in csv_files))

2013 files: 343
2014 files: 382


In [24]:
row[f"{col}_minus7999_count"] = (values == -7999).sum()


In [25]:
from pathlib import Path

folder = Path("/content/solar_data/data/raw/system_1283")

csv_files = sorted(
    file for file in folder.rglob("*.csv")
    if "__date_2013_" in file.name
    or "__date_2014_" in file.name
)

print("2013 files:", sum("__date_2013_" in f.name for f in csv_files))
print("2014 files:", sum("__date_2014_" in f.name for f in csv_files))

2013 files: 343
2014 files: 382


In [26]:
!aws s3 cp "s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2013/" "/content/solar_data/data/raw/system_1283/year_2013/" --recursive --exclude "*" --include "*.csv" --no-sign-request

download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2013/month=1/day=1/system_1283__date_2013_01_01.csv to solar_data/data/raw/system_1283/year_2013/month=1/day=1/system_1283__date_2013_01_01.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2013/month=1/day=12/system_1283__date_2013_01_12.csv to solar_data/data/raw/system_1283/year_2013/month=1/day=12/system_1283__date_2013_01_12.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2013/month=1/day=18/system_1283__date_2013_01_18.csv to solar_data/data/raw/system_1283/year_2013/month=1/day=18/system_1283__date_2013_01_18.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2013/month=1/day=16/system_1283__date_2013_01_16.csv to solar_data/data/raw/system_1283/year_2013/month=1/day=16/system_1283__date_2013_01_16.csv
download: s3://oedi-data-lake/pvdaq/csv/pvdata/system_id=1283/year=2013/month=1/day=2/system_1283__date_2013_01_02.csv to solar_data/data/raw/system_1

In [27]:
results = []

Generate the Verified 2013–2014 Quality Report

In [28]:
from pathlib import Path
import hashlib
import pandas as pd
from google.colab import files

# Folder used by your AWS download commands.
FOLDER = Path("/content/solar_data/data/raw/system_1283")
YEARS = {2013, 2014}

REQUIRED = [
    "ac_power_metered_kw__1040",
    "poa_irradiance__1055",
    "ambient_temp__1053"
]

# --------------------------------------------------
# 1. Find the selected years' files.
# --------------------------------------------------
csv_files = sorted(
    file for file in FOLDER.rglob("*.csv")
    if any(f"__date_{year}_" in file.name for year in YEARS)
)

for year in sorted(YEARS):
    count = sum(f"__date_{year}_" in file.name for file in csv_files)
    print(f"{year}: {count} CSV file copies found")

    if count == 0:
        raise ValueError(
            f"No files found for {year}. Download that year into "
            f"{FOLDER}/year_{year}/ before running this cell again."
        )

# --------------------------------------------------
# 2. Inspect every file, one at a time.
# --------------------------------------------------
results = []

for file in csv_files:
    file_date = pd.to_datetime(
        file.stem.split("__date_")[-1],
        format="%Y_%m_%d"
    )

    row = {
        "date": file_date,
        "file": file.name,
        "file_path": str(file),
        "read_error": ""
    }

    try:
        # Hash identifies byte-for-byte identical file copies.
        with file.open("rb") as stream:
            digest = hashlib.sha256()
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)

        row["sha256"] = digest.hexdigest()

        df = pd.read_csv(file)

        row["rows"] = len(df)
        row["exact_duplicate_rows"] = int(df.duplicated().sum())

        missing_columns = [
            col for col in ["measured_on"] + REQUIRED
            if col not in df.columns
        ]
        row["missing_columns"] = ", ".join(missing_columns)

        # Timestamp checks.
        if "measured_on" in df.columns:
            timestamps = pd.to_datetime(
                df["measured_on"], errors="coerce"
            )
            valid_times = timestamps.dropna()

            row["invalid_timestamps"] = int(timestamps.isna().sum())
            row["duplicate_timestamps"] = int(
                valid_times.duplicated().sum()
            )
            row["first_time"] = valid_times.min()
            row["last_time"] = valid_times.max()
            row["timestamps_on_other_dates"] = int(
                (valid_times.dt.normalize() != file_date).sum()
            )

            intervals = (
                valid_times.drop_duplicates()
                .sort_values()
                .diff()
                .dt.total_seconds()
                .dropna()
            )

            typical_interval = intervals.median()

            row["median_interval_seconds"] = typical_interval
            row["largest_gap_seconds"] = intervals.max()

            # Diagnostic count; not an automatic cleaning rule.
            row["gaps_above_1_5x_median"] = (
                int((intervals > 1.5 * typical_interval).sum())
                if pd.notna(typical_interval)
                else None
            )

        # Measurement checks.
        for col in REQUIRED:
            if col not in df.columns:
                continue

            values = pd.to_numeric(df[col], errors="coerce")

            row[f"{col}_missing_pct"] = (
                values.isna().mean() * 100
            )
            row[f"{col}_nonnumeric_count"] = int(
                (df[col].notna() & values.isna()).sum()
            )
            row[f"{col}_minus7999_count"] = int(
                (values == -7999).sum()
            )
            row[f"{col}_negative_count"] = int(
                (values < 0).sum()
            )
            row[f"{col}_min"] = values.min()
            row[f"{col}_max"] = values.max()

    except Exception as error:
        row["read_error"] = str(error)

    # Append once for each inspected file, including failed reads.
    results.append(row)

# --------------------------------------------------
# 3. Build and verify the report.
# --------------------------------------------------
if not results:
    raise ValueError("No inspection results were produced.")

quality = pd.DataFrame(results).sort_values(
    ["date", "file_path"]
).reset_index(drop=True)

actual_years = set(quality["date"].dt.year)

if actual_years != YEARS:
    raise ValueError(f"Unexpected years in report: {actual_years}")

# Flag repeated file names; do not delete any raw files.
quality["copies_with_same_filename"] = (
    quality.groupby("file")["file"].transform("size")
)

print("\nFile copies inspected by year:")
print(quality["date"].dt.year.value_counts().sort_index())

print("\nRead errors:", quality["read_error"].ne("").sum())

repeated = quality[
    quality["copies_with_same_filename"] > 1
]

print("\nEntries with repeated filenames:", len(repeated))

if not repeated.empty:
    display(repeated[["file", "file_path", "sha256"]])

# --------------------------------------------------
# 4. Check dates without any file.
# --------------------------------------------------
expected_dates = pd.date_range("2013-01-01", "2014-12-31")
available_dates = pd.DatetimeIndex(quality["date"].unique())
missing_dates = expected_dates.difference(available_dates)

print("\nExpected calendar days:", len(expected_dates))
print("Dates with files:", len(available_dates))
print("Dates without files:", len(missing_dates))

missing_report = pd.DataFrame({"missing_date": missing_dates})

display(missing_report)
display(quality.head())

# --------------------------------------------------
# 5. Save and download the new quality report.
# --------------------------------------------------
report_path = "/content/solar_quality_report_2013_2014_checked.csv"
missing_path = "/content/solar_missing_dates_2013_2014.csv"

quality.to_csv(report_path, index=False)
missing_report.to_csv(missing_path, index=False)

print("\nSaved:", report_path)
print("Saved:", missing_path)

files.download(report_path)

2013: 343 CSV file copies found
2014: 382 CSV file copies found

File copies inspected by year:
date
2013    343
2014    382
Name: count, dtype: int64

Read errors: 0

Entries with repeated filenames: 62


,file,file_path,sha256
343,system_1283__date_2014_01_01.csv,/content/solar_data/data/raw/system_1283/year_...,cc0286900436230b4b53c7216a5b821b2a85701dd804fb...
344,system_1283__date_2014_01_01.csv,/content/solar_data/data/raw/system_1283/year_...,cc0286900436230b4b53c7216a5b821b2a85701dd804fb...
345,system_1283__date_2014_01_02.csv,/content/solar_data/data/raw/system_1283/year_...,f55da7fa5498e2b77c029957c3e9bd752c6d3cc7b507b7...
346,system_1283__date_2014_01_02.csv,/content/solar_data/data/raw/system_1283/year_...,f55da7fa5498e2b77c029957c3e9bd752c6d3cc7b507b7...
347,system_1283__date_2014_01_03.csv,/content/solar_data/data/raw/system_1283/year_...,76d9bd31050fd077e73da1f4b54c7b550959813374246a...
...,...,...,...
400,system_1283__date_2014_01_29.csv,/content/solar_data/data/raw/system_1283/year_...,aa5ecef187b53ae945e9d90ff18c6e74e0d0c8e16b47cd...
401,system_1283__date_2014_01_30.csv,/content/solar_data/data/raw/system_1283/year_...,7be10555321e8e5ed8acdf746868282c95dee68b143bbb...
402,system_1283__date_2014_01_30.csv,/content/solar_data/data/raw/system_1283/year_...,7be10555321e8e5ed8acdf746868282c95dee68b143bbb...
403,system_1283__date_2014_01_31.csv,/content/solar_data/data/raw/system_1283/year_...,19f564555fcda0478b9795131d8941330dcb26ccf58bb1...



Expected calendar days: 730
Dates with files: 694
Dates without files: 36


,missing_date
0,2013-01-05
1,2013-01-08
2,2013-01-13
3,2013-01-19
4,2013-01-24
5,2013-01-25
6,2013-01-26
7,2013-01-29
8,2013-01-30
9,2013-02-01


,date,file,file_path,read_error,sha256,rows,exact_duplicate_rows,missing_columns,invalid_timestamps,duplicate_timestamps,...,poa_irradiance__1055_negative_count,poa_irradiance__1055_min,poa_irradiance__1055_max,ambient_temp__1053_missing_pct,ambient_temp__1053_nonnumeric_count,ambient_temp__1053_minus7999_count,ambient_temp__1053_negative_count,ambient_temp__1053_min,ambient_temp__1053_max,copies_with_same_filename
0,2013-01-01,system_1283__date_2013_01_01.csv,/content/solar_data/data/raw/system_1283/year_...,,62186fa47814762ec3ef93e664db21f3b8dcd28e3f98f0...,5760,0,,0,0,...,0,0.0,502.3807,0.0,0,0,5316,-11.4405,0.9768,1
1,2013-01-02,system_1283__date_2013_01_02.csv,/content/solar_data/data/raw/system_1283/year_...,,7375aa592c49994c8141379cdd82e02b55b1d24456f585...,5760,0,,0,0,...,0,0.0,740.9064,0.0,0,0,4723,-8.9901,3.9194,1
2,2013-01-03,system_1283__date_2013_01_03.csv,/content/solar_data/data/raw/system_1283/year_...,,2de1ff29cd226d791e584f2640d74d18e72297c3331bb9...,5760,0,,0,0,...,0,0.0,516.2969,0.0,0,0,4054,-7.9595,5.7990,1
3,2013-01-04,system_1283__date_2013_01_04.csv,/content/solar_data/data/raw/system_1283/year_...,,b8a84a4dd8defcc5e98fb043946e6aa2c09c3715d3e9d4...,5760,0,,0,0,...,0,0.0,505.3711,0.0,0,0,3199,-3.6483,12.4635,1
4,2013-01-06,system_1283__date_2013_01_06.csv,/content/solar_data/data/raw/system_1283/year_...,,11318dc847306dbb137a6c5005be58406bc8032ded3003...,5760,0,,0,0,...,0,0.0,571.5333,0.0,0,0,2088,-5.2761,13.7841,1



Saved: /content/solar_quality_report_2013_2014_checked.csv
Saved: /content/solar_missing_dates_2013_2014.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Next Stage: Sensor-Metadata Verification

In [29]:
%pip -q install pyarrow

In [30]:
from pathlib import Path
from io import BytesIO
import json
import requests
import pandas as pd

BASE = "https://oedi-data-lake.s3.amazonaws.com/"

metadata_folder = Path("/content/solar_data/metadata/system_1283")
metadata_folder.mkdir(parents=True, exist_ok=True)

# Download and preserve the original system metadata.
system_url = (
    BASE
    + "pvdaq/csv/system_metadata/1283_system_metadata.json"
)

response = requests.get(system_url, timeout=60)
response.raise_for_status()

(metadata_folder / "1283_system_metadata.json").write_bytes(
    response.content
)

system_metadata = response.json()

print("SYSTEM METADATA")
print(json.dumps(system_metadata, indent=2))

# Download and preserve the original sensor metadata.
sensor_url = (
    BASE
    + "pvdaq/parquet/metrics/"
    + "metrics__system_1283__part000.parquet"
)

response = requests.get(sensor_url, timeout=60)
response.raise_for_status()

(metadata_folder / "1283_sensor_metadata.parquet").write_bytes(
    response.content
)

metrics = pd.read_parquet(BytesIO(response.content))

print("\nAvailable metadata columns:")
print(metrics.columns.tolist())

if "standard_name" not in metrics.columns:
    raise ValueError("Sensor-name field is missing. Inspect the columns above.")

selected_sensors = [
    "ac_power_metered_kw__1040",
    "ac_power__1137",
    "ac_meter_1_power_kw__1041",
    "ac_meter_2_power_kw__1042",
    "inv1_ac_power_kw__1043",
    "inv2_ac_power_kw__1047",
    "poa_irradiance__1055",
    "poa_irradiance_refcell__1054",
    "ambient_temp__1053",
    "module_temp__1056",
    "wind_speed__1051"
]

sensor_check = metrics.loc[
    metrics["standard_name"].isin(selected_sensors)
].copy()

print("\nSELECTED SENSOR DEFINITIONS")
display(sensor_check)

not_found = set(selected_sensors) - set(sensor_check["standard_name"])
print("\nSensors not found:", sorted(not_found))

sensor_check.to_csv(
    metadata_folder / "1283_selected_sensor_metadata.csv",
    index=False
)

SYSTEM METADATA
{
  "System": {
    "system_id": 1283,
    "site_id": 1271,
    "public_name": "[1283] NREL Research Support Facility II",
    "area": "2115.84",
    "power": "408.24",
    "started_on": "2012-01-10 00:00:00",
    "ended_on": "",
    "comments": "NREL-owned arrays"
  },
  "Site": {
    "site_id": "1271",
    "public_name": "[1283] NREL Research Support Facility II",
    "latitude": "39.7409",
    "longitude": "-105.1711",
    "elevation": "1800.0",
    "climate_type": "Bsk",
    "av_temp": "12.5",
    "av_pressure": "820.0"
  },
  "Mount": {
    "Mount 0": {
      "mount_id": 1274,
      "name": "nrsfi_1283_mt1",
      "manufacturer": "",
      "model": "",
      "tracking": "f",
      "azimuth": "165.0",
      "tilt": "10.0"
    }
  },
  "Inverters": {
    "Inverter 0": {
      "inverter_id": "1261",
      "name": "nrresufaii_1283_inv_0",
      "manufacturer": "SMA",
      "model": "Sunny Central 250U",
      "type": "central",
      "quantity": "2",
      "serial_num"

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
3,1283,1137,ac_power,AC power,W,W,1.0,0.0,ac_meter_1_power_kW+ac_meter_2_power_kW,avg,None,NaN,,ac_power__1137
6,1283,1040,ac_power_metered_kW,AC power,kW,W,1000.0,0.0,,avg,None,NaN,,ac_power_metered_kw__1040
7,1283,1041,ac_meter_1_power_kW,AC power,kW,W,1000.0,0.0,,avg,None,NaN,,ac_meter_1_power_kw__1041
8,1283,1042,ac_meter_2_power_kW,AC power,kW,W,1000.0,0.0,,avg,None,NaN,,ac_meter_2_power_kw__1042
9,1283,1043,inv1_ac_power_kW,AC power,kW,W,1000.0,0.0,,avg,None,NaN,,inv1_ac_power_kw__1043
10,1283,1047,inv2_ac_power_kW,AC power,kW,W,1000.0,0.0,,avg,None,NaN,,inv2_ac_power_kw__1047
11,1283,1055,POA_irradiance,Irradiance POA,W/m^2,W/m^2,1.0,0.0,,avg,None,NaN,,poa_irradiance__1055
12,1283,1054,POA_irradiance_refcell,Irradiance POA,W/m^2,W/m^2,1.0,0.0,,avg,None,NaN,,poa_irradiance_refcell__1054
13,1283,1056,module_temp,Temperature module,C,C,1.0,0.0,,avg,None,NaN,,module_temp__1056
14,1283,1053,ambient_temp,Temperature ambient,C,C,1.0,0.0,,avg,None,NaN,,ambient_temp__1053



Sensors not found: []


In [31]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/solar_1283_metadata",
    "zip",
    "/content/solar_data/metadata/system_1283"
)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
from pathlib import Path
import pandas as pd

folder = Path("/content/solar_data/data/raw/system_1283")
matches = sorted(
    folder.rglob("system_1283__date_2013_01_01.csv")
)

if not matches:
    raise ValueError("The selected sample file was not found.")

df = pd.read_csv(matches[0])

power_columns = [
    "ac_power_metered_kw__1040",
    "ac_power__1137",
    "ac_meter_1_power_kw__1041",
    "ac_meter_2_power_kw__1042",
    "inv1_ac_power_kw__1043",
    "inv2_ac_power_kw__1047"
]

missing = set(power_columns) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

power = df[power_columns].apply(pd.to_numeric, errors="coerce")

print("Power ranges:")
display(power.describe().loc[["min", "50%", "max"]])

# Compare overlapping measurements without changing them.
meter_sum = (
    power["ac_meter_1_power_kw__1041"]
    + power["ac_meter_2_power_kw__1042"]
)

# Use positive-generation readings for meaningful ratios.
mask = meter_sum > 10

if not mask.any():
    raise ValueError("No suitable positive-generation readings in this sample.")

comparison = pd.DataFrame({
    "metered_power": power["ac_power_metered_kw__1040"],
    "calculated_power": power["ac_power__1137"],
    "sum_of_two_meters": meter_sum
})

print("Example readings during generation:")
display(comparison.loc[mask].head(10))

print(
    "Median calculated_power / meter_sum:",
    (power.loc[mask, "ac_power__1137"] / meter_sum.loc[mask]).median()
)

print(
    "Median metered_power / meter_sum:",
    (power.loc[mask, "ac_power_metered_kw__1040"]
     / meter_sum.loc[mask]).median()
)

Power ranges:


,ac_power_metered_kw__1040,ac_power__1137,ac_meter_1_power_kw__1041,ac_meter_2_power_kw__1042,inv1_ac_power_kw__1043,inv2_ac_power_kw__1047
min,0.0,0.0,0.000,0.0000,0.0,0.0000
50%,0.0,0.0,0.000,0.0000,0.0,0.0000
max,234.5,235838.1,123.712,114.7573,0.0,114.0534


Example readings during generation:


,metered_power,calculated_power,sum_of_two_meters
2148,9.9,10042.2,10.0422
2149,10.0,10165.2,10.1652
2150,10.1,10254.7,10.2547
2151,10.2,10275.6,10.2756
2152,10.2,10302.7,10.3027
2153,10.1,10210.9,10.2109
2154,10.2,10231.9,10.2319
2155,10.0,10152.6,10.1526
2156,10.0,10288.1,10.2881
2157,10.2,10265.1,10.2651


Median calculated_power / meter_sum: 1000.0
Median metered_power / meter_sum: 0.9954606992115951


In [33]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

# ---------- SETTINGS ----------
FOLDER = Path("/content/solar_data/data/raw/system_1283")
YEARS = {2013, 2014}

COLUMNS = {
    "metered": "ac_power_metered_kw__1040",
    "calculated": "ac_power__1137",
    "meter1": "ac_meter_1_power_kw__1041",
    "meter2": "ac_meter_2_power_kw__1042",
    "inverter1": "inv1_ac_power_kw__1043",
}

# This threshold selects generation readings for comparison.
# It does not remove readings from your original data.
GENERATION_THRESHOLD_KW = 10

# ---------- FIND FILES ----------
candidates = sorted(FOLDER.rglob("system_1283__date_*.csv"))

files = []

for path in candidates:
    try:
        file_date = pd.to_datetime(
            path.stem.split("__date_")[1],
            format="%Y_%m_%d"
        )
    except (ValueError, IndexError):
        continue

    if file_date.year in YEARS:
        files.append((path, file_date))

if not files:
    raise FileNotFoundError(
        f"No 2013 or 2014 daily CSV files found under {FOLDER}"
    )

# ---------- CHECK REPEATED DOWNLOADS ----------
def file_hash(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


seen_names = {}
unique_files = []
repeated_copies = 0

for path, file_date in files:
    digest = file_hash(path)

    if path.name in seen_names:
        previous_digest, previous_path = seen_names[path.name]

        if digest != previous_digest:
            raise ValueError(
                "Two files have the same name but different contents:\n"
                f"{previous_path}\n{path}\n"
                "Investigate these before continuing."
            )

        # Analyze identical copies once; retain all original files.
        repeated_copies += 1
        continue

    seen_names[path.name] = (digest, path)
    unique_files.append((path, file_date))

print("Unique daily files:", len(unique_files))
print("Identical repeated copies:", repeated_copies)

# ---------- ANALYZE EACH DAY ----------
results = []

for path, file_date in unique_files:
    row = {
        "date": file_date.date(),
        "year": file_date.year,
        "file": str(path),
        "error": "",
    }

    try:
        df = pd.read_csv(path)

        required = ["measured_on", *COLUMNS.values()]
        missing = [column for column in required if column not in df.columns]

        if missing:
            raise ValueError(f"Missing columns: {missing}")

        power = df[list(COLUMNS.values())].apply(
            pd.to_numeric, errors="coerce"
        )

        metered = power[COLUMNS["metered"]]
        calculated_kw = power[COLUMNS["calculated"]] / 1000
        meter1 = power[COLUMNS["meter1"]]
        meter2 = power[COLUMNS["meter2"]]
        inverter1 = power[COLUMNS["inverter1"]]

        meter_sum = meter1 + meter2

        # Compare only readings where both meters are available,
        # nonnegative, and their combined output exceeds 10 kW.
        generation = (
            meter1.notna()
            & meter2.notna()
            & meter1.ge(0)
            & meter2.ge(0)
            & meter_sum.gt(GENERATION_THRESHOLD_KW)
        )

        valid_calculated = (
            generation
            & calculated_kw.notna()
            & calculated_kw.ge(0)
        )

        valid_metered = (
            generation
            & metered.notna()
            & metered.ge(0)
        )

        calculated_error = (
            calculated_kw[valid_calculated]
            - meter_sum[valid_calculated]
        )

        metered_error = (
            metered[valid_metered]
            - meter_sum[valid_metered]
        )

        row["rows"] = len(df)
        row["generation_comparison_rows"] = int(generation.sum())
        row["calculated_comparison_rows"] = int(valid_calculated.sum())
        row["metered_comparison_rows"] = int(valid_metered.sum())

        row["calculated_kw_to_meter_sum_median"] = (
            calculated_kw[valid_calculated]
            / meter_sum[valid_calculated]
        ).median()

        row["calculated_vs_sum_mae_kw"] = (
            calculated_error.abs().mean()
        )

        row["metered_to_meter_sum_median"] = (
            metered[valid_metered]
            / meter_sum[valid_metered]
        ).median()

        row["metered_vs_sum_mae_kw"] = metered_error.abs().mean()

        row["metered_vs_sum_median_difference_pct"] = (
            100 * metered_error / meter_sum[valid_metered]
        ).median()

        # Check inverter 1 specifically when meter 1 reports generation.
        inverter_check = (
            meter1.gt(GENERATION_THRESHOLD_KW)
            & inverter1.notna()
            & inverter1.ge(0)
        )

        row["inverter1_comparison_rows"] = int(inverter_check.sum())

        row["inverter1_zero_pct_when_meter1_generates"] = (
            100 * inverter1[inverter_check].eq(0).mean()
        )

        row["inverter1_missing_pct"] = (
            100 * inverter1.isna().mean()
        )

        # Timestamp checks: parsing does not establish the true timezone.
        timestamps = pd.to_datetime(
            df["measured_on"], errors="coerce"
        )

        valid_times = timestamps.dropna()
        ordered_times = valid_times.drop_duplicates().sort_values()
        intervals = ordered_times.diff().dt.total_seconds().dropna()

        row["invalid_timestamps"] = int(timestamps.isna().sum())
        row["duplicate_timestamps"] = int(valid_times.duplicated().sum())
        row["timestamps_in_original_order"] = (
            valid_times.is_monotonic_increasing
        )
        row["median_interval_seconds"] = intervals.median()
        row["max_interval_seconds"] = intervals.max()
        row["first_timestamp"] = (
            str(ordered_times.iloc[0]) if len(ordered_times) else ""
        )
        row["last_timestamp"] = (
            str(ordered_times.iloc[-1]) if len(ordered_times) else ""
        )

    except Exception as error:
        row["error"] = str(error)

    results.append(row)

report = pd.DataFrame(results).sort_values("date")

# ---------- DISPLAY RESULTS ----------
successful = report.loc[report["error"].eq("")].copy()

print("\nFiles with errors:", int(report["error"].ne("").sum()))

if report["error"].ne("").any():
    display(report.loc[report["error"].ne(""), ["date", "error"]])

if successful.empty:
    raise ValueError("No files could be analyzed. See errors above.")

summary = successful.groupby("year").agg(
    days_checked=("date", "count"),
    calculated_ratio_median=(
        "calculated_kw_to_meter_sum_median", "median"
    ),
    calculated_daily_mae_median_kw=(
        "calculated_vs_sum_mae_kw", "median"
    ),
    metered_ratio_median=(
        "metered_to_meter_sum_median", "median"
    ),
    metered_daily_mae_median_kw=(
        "metered_vs_sum_mae_kw", "median"
    ),
    inverter1_daily_zero_pct_median=(
        "inverter1_zero_pct_when_meter1_generates", "median"
    ),
    typical_interval_seconds=(
        "median_interval_seconds", "median"
    ),
)

print("\nYEAR SUMMARY — medians of daily diagnostic results:")
display(summary.round(4))

successful["month"] = pd.to_datetime(
    successful["date"]
).dt.to_period("M").astype(str)

monthly = successful.groupby("month").agg(
    days_checked=("date", "count"),
    metered_ratio_median=(
        "metered_to_meter_sum_median", "median"
    ),
    metered_daily_mae_median_kw=(
        "metered_vs_sum_mae_kw", "median"
    ),
    inverter1_daily_zero_pct_median=(
        "inverter1_zero_pct_when_meter1_generates", "median"
    ),
)

print("\nMONTHLY SUMMARY:")
display(monthly.round(4))

print("\nTIMESTAMP INTERVALS:")
display(
    successful.groupby(["year", "median_interval_seconds"])
    .size()
    .rename("days")
    .reset_index()
)

print("\nDays with the largest meter differences:")
display(
    successful.nlargest(10, "metered_vs_sum_mae_kw")[
        [
            "date",
            "metered_comparison_rows",
            "metered_to_meter_sum_median",
            "metered_vs_sum_mae_kw",
            "inverter1_zero_pct_when_meter1_generates",
        ]
    ]
)

# ---------- SAVE DIAGNOSTIC REPORT ----------
OUTPUT = Path("/content/solar_data/reports")
OUTPUT.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT / "sensor_verification_2013_2014.csv"
report.to_csv(output_file, index=False)

print("\nReport saved to:", output_file)

Unique daily files: 694
Identical repeated copies: 31

Files with errors: 3


,date,error
591,2014-06-14,"Missing columns: ['ac_power__1137', 'ac_meter_..."
606,2014-06-28,"Missing columns: ['ac_power__1137', 'ac_meter_..."
645,2014-08-02,"Missing columns: ['ac_power_metered_kw__1040',..."



YEAR SUMMARY — medians of daily diagnostic results:


,days_checked,calculated_ratio_median,calculated_daily_mae_median_kw,metered_ratio_median,metered_daily_mae_median_kw,inverter1_daily_zero_pct_median,typical_interval_seconds
year,,,,,,,
2013,343,1.0,0.0,0.9975,0.8529,100.0,15.0
2014,348,1.0,0.0,0.9978,0.8987,100.0,15.0



MONTHLY SUMMARY:


...


TIMESTAMP INTERVALS:


,year,median_interval_seconds,days
0,2013,15.0,343
1,2014,15.0,348



Days with the largest meter differences:


,date,metered_comparison_rows,metered_to_meter_sum_median,metered_vs_sum_mae_kw,inverter1_zero_pct_when_meter1_generates
471,2014-02-14,2329.0,0.997320,5.492391,100.0
563,2014-05-17,3200.0,0.998567,5.003899,100.0
203,2013-05-21,3160.0,0.997094,4.709218,100.0
478,2014-02-20,2358.0,0.996845,4.170935,100.0
163,2013-04-12,2725.0,0.996849,3.928844,100.0
52,2013-11-01,2229.0,0.996939,3.607450,100.0
249,2013-06-08,3184.0,0.997409,3.389014,100.0
162,2013-04-11,1743.0,0.995342,3.351685,100.0
548,2014-04-30,3074.0,0.997266,3.304809,100.0
561,2014-05-15,3160.0,0.997479,3.205815,100.0



Report saved to: /content/solar_data/reports/sensor_verification_2013_2014.csv


Verify timestamp and measurement conventions

In [34]:
from pathlib import Path
import json
import re
import requests
import pandas as pd
from IPython.display import display

# ---------- LOCATIONS ----------
PROJECT = Path("/content/solar_data")
RAW = PROJECT / "data/raw/system_1283"
METADATA = PROJECT / "data/raw/metadata"
METADATA.mkdir(parents=True, exist_ok=True)

URL = (
    "https://oedi-data-lake.s3.amazonaws.com/"
    "pvdaq/csv/system_metadata/1283_system_metadata.json"
)

# ---------- DOWNLOAD SYSTEM METADATA ----------
response = requests.get(URL, timeout=60)
response.raise_for_status()

metadata_file = METADATA / "1283_system_metadata.json"
metadata_file.write_bytes(response.content)

system_metadata = json.loads(response.content)

print("Metadata saved to:", metadata_file)

# ---------- FIND TIME-RELATED FIELDS ----------
time_fields = []

def find_time_fields(value, location="root"):
    if isinstance(value, dict):
        for key, item in value.items():
            field_path = f"{location}.{key}"

            if re.search(
                r"time|interval|frequency|sampling|daylight|utc|offset",
                str(key),
                flags=re.IGNORECASE
            ):
                time_fields.append({
                    "field": field_path,
                    "value": json.dumps(item, ensure_ascii=False)
                })

            find_time_fields(item, field_path)

    elif isinstance(value, list):
        for index, item in enumerate(value):
            find_time_fields(item, f"{location}[{index}]")

find_time_fields(system_metadata)

print("\nTIME-RELATED METADATA FIELDS:")

if time_fields:
    display(pd.DataFrame(time_fields))
else:
    print("No time-related fields were found.")

print(
    "\nNote: a field named 'offset' may describe sensor scaling, "
    "rather than timezone. Read its surrounding context."
)

# ---------- SHOW SAMPLE RAW TIMESTAMPS ----------
sample_files = sorted(
    RAW.rglob("system_1283__date_2013_01_01.csv")
)

if not sample_files:
    raise FileNotFoundError(
        "The January 1, 2013 file was not found. "
        "Check the RAW folder path."
    )

sample = pd.read_csv(sample_files[0], nrows=10)

if "measured_on" not in sample.columns:
    raise ValueError("Sample file has no measured_on column.")

timestamp_text = sample["measured_on"].astype(str)

print("\nSAMPLE TIMESTAMPS:")
display(sample[["measured_on"]])

# Check for common explicit UTC/offset suffixes.
has_offset = timestamp_text.str.contains(
    r"(?:Z|[+-]\d{2}:?\d{2})$",
    regex=True,
    case=False
)

print(
    "\nSample timestamps contain an explicit UTC/offset suffix:",
    bool(has_offset.any())
)

print(
    "\nIf timestamps have no offset, their timezone cannot "
    "be established from their text alone."
)

# ---------- DISPLAY FULL METADATA FOR INSPECTION ----------
print("\nFULL SYSTEM METADATA:")
print(json.dumps(system_metadata, indent=2, ensure_ascii=False))

Metadata saved to: /content/solar_data/data/raw/metadata/1283_system_metadata.json

TIME-RELATED METADATA FIELDS:


,field,value
0,root.Inverters.Inverter 0.time_interval,""""""



Note: a field named 'offset' may describe sensor scaling, rather than timezone. Read its surrounding context.

SAMPLE TIMESTAMPS:


,measured_on
0,2013-01-01 00:00:00
1,2013-01-01 00:00:15
2,2013-01-01 00:00:30
3,2013-01-01 00:00:45
4,2013-01-01 00:01:00
5,2013-01-01 00:01:15
6,2013-01-01 00:01:30
7,2013-01-01 00:01:45
8,2013-01-01 00:02:00
9,2013-01-01 00:02:15



Sample timestamps contain an explicit UTC/offset suffix: False

If timestamps have no offset, their timezone cannot be established from their text alone.

FULL SYSTEM METADATA:
{
  "System": {
    "system_id": 1283,
    "site_id": 1271,
    "public_name": "[1283] NREL Research Support Facility II",
    "area": "2115.84",
    "power": "408.24",
    "started_on": "2012-01-10 00:00:00",
    "ended_on": "",
    "comments": "NREL-owned arrays"
  },
  "Site": {
    "site_id": "1271",
    "public_name": "[1283] NREL Research Support Facility II",
    "latitude": "39.7409",
    "longitude": "-105.1711",
    "elevation": "1800.0",
    "climate_type": "Bsk",
    "av_temp": "12.5",
    "av_pressure": "820.0"
  },
  "Mount": {
    "Mount 0": {
      "mount_id": 1274,
      "name": "nrsfi_1283_mt1",
      "manufacturer": "",
      "model": "",
      "tracking": "f",
      "azimuth": "165.0",
      "tilt": "10.0"
    }
  },
  "Inverters": {
    "Inverter 0": {
      "inverter_id": "1261",
      "na

Standardize variables and add quality flags

In [35]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

PROJECT = Path("/content/solar_data")

REPORT_FILE = (
    PROJECT / "reports/sensor_verification_2013_2014.csv"
)

OUTPUT = PROJECT / "data/processed/system_1283"
OUTPUT.mkdir(parents=True, exist_ok=True)

if not REPORT_FILE.exists():
    raise FileNotFoundError(
        f"Report not found: {REPORT_FILE}. "
        "Run the sensor-verification cell first."
    )

report = pd.read_csv(REPORT_FILE)

if report["date"].duplicated().any():
    raise ValueError(
        "The report contains repeated dates. "
        "Resolve these before preparing the working dataset."
    )

# New name: (original column, conversion factor)
VARIABLES = {
    "ac_power_kw": ("ac_power_metered_kw__1040", 1),
    "ac_power_calculated_kw": ("ac_power__1137", 0.001),
    "meter1_power_kw": ("ac_meter_1_power_kw__1041", 1),
    "meter2_power_kw": ("ac_meter_2_power_kw__1042", 1),
    "inverter1_power_kw": ("inv1_ac_power_kw__1043", 1),
    "inverter2_power_kw": ("inv2_ac_power_kw__1047", 1),
    "poa_irradiance_w_m2": ("poa_irradiance__1055", 1),
    "ambient_temp_c": ("ambient_temp__1053", 1),
    "module_temp_c": ("module_temp__1056", 1),
    "wind_speed_m_s": ("wind_speed__1051", 1),
}

manifest = []

for index, record in enumerate(report.itertuples(index=False), start=1):
    source = Path(record.file)

    result = {
        "date": record.date,
        "source_file": str(source),
        "output_file": "",
        "rows": None,
        "absent_columns": "",
        "error": "",
    }

    try:
        df = pd.read_csv(source)

        if "measured_on" not in df.columns:
            raise ValueError("measured_on column is absent")

        # Retain all original columns.
        working = df.copy()

        timestamps = pd.to_datetime(
            df["measured_on"], errors="coerce"
        )

        working["timestamp_source_local"] = timestamps
        working["source_local_date"] = (
            timestamps.dt.strftime("%Y-%m-%d")
        )
        working["flag_invalid_timestamp"] = timestamps.isna()
        working["flag_duplicate_timestamp"] = (
            timestamps.notna()
            & timestamps.duplicated(keep=False)
        )
        working["flag_date_differs_from_filename"] = (
            timestamps.notna()
            & working["source_local_date"].ne(str(record.date))
        )

        absent = []

        for new_name, (original_name, factor) in VARIABLES.items():

            if original_name in df.columns:
                original = df[original_name]

                numeric = pd.to_numeric(
                    original, errors="coerce"
                )

                # Flag -7999 as a suspected sentinel.
                # Preserve it until its treatment is documented.
                suspected_sentinel = numeric.eq(-7999)

                nonnumeric = (
                    original.notna() & numeric.isna()
                )

            else:
                absent.append(original_name)

                numeric = pd.Series(
                    float("nan"), index=df.index
                )
                suspected_sentinel = pd.Series(
                    False, index=df.index
                )
                nonnumeric = pd.Series(
                    False, index=df.index
                )

            working[new_name] = numeric * factor

            working[f"flag_{new_name}_missing"] = numeric.isna()
            working[f"flag_{new_name}_nonnumeric"] = nonnumeric
            working[f"flag_{new_name}_suspected_sentinel"] = (
                suspected_sentinel
            )

        # Diagnostic flag, not a confirmed equipment fault.
        working["flag_inverter1_zero_while_meter1_generates"] = (
            working["meter1_power_kw"].gt(10)
            & working["inverter1_power_kw"].eq(0)
        )

        output_file = (
            OUTPUT / f"system_1283__date_{record.date}.csv"
        )

        working.to_csv(output_file, index=False)

        result["output_file"] = str(output_file)
        result["rows"] = len(working)
        result["absent_columns"] = "; ".join(absent)

    except Exception as error:
        result["error"] = str(error)

    manifest.append(result)

    if index % 100 == 0:
        print(f"Prepared {index} of {len(report)} files")

manifest = pd.DataFrame(manifest)

manifest_file = OUTPUT / "preparation_manifest.csv"
manifest.to_csv(manifest_file, index=False)

# Record verified facts and unresolved conventions.
notes = {
    "system_id": 1283,
    "years": [2013, 2014],
    "timestamp_basis": "instrument local time",
    "timezone_and_dst_convention": "unverified",
    "typical_observed_spacing_seconds": 15,
    "measurement_interval_boundaries": "unverified",
    "ac_power_kw_source": "ac_power_metered_kw__1040",
    "calculated_power_conversion": "ac_power__1137 divided by 1000",
    "suspected_sentinel": -7999,
    "sentinel_treatment": "flagged and retained",
    "original_columns": "retained",
    "daily_energy": "not calculated at this stage",
    "documentation": (
        "https://github.com/openEDI/documentation/blob/main/pvdaq.md"
    ),
}

(OUTPUT / "preparation_notes.json").write_text(
    json.dumps(notes, indent=2),
    encoding="utf-8"
)

print("\nPreparation summary:")
print("Files listed:", len(manifest))
print("Files prepared:", int(manifest["error"].eq("").sum()))
print("Files with errors:", int(manifest["error"].ne("").sum()))
print("Output folder:", OUTPUT)

display(manifest.head())

if manifest["error"].ne("").any():
    print("\nFILES REQUIRING INVESTIGATION:")
    display(
        manifest.loc[
            manifest["error"].ne(""),
            ["date", "error"]
        ]
    )

Prepared 100 of 694 files
Prepared 200 of 694 files
Prepared 300 of 694 files
Prepared 400 of 694 files
Prepared 500 of 694 files
Prepared 600 of 694 files

Preparation summary:
Files listed: 694
Files prepared: 694
Files with errors: 0
Output folder: /content/solar_data/data/processed/system_1283


,date,source_file,output_file,rows,absent_columns,error
0,2013-01-01,/content/solar_data/data/raw/system_1283/year_...,/content/solar_data/data/processed/system_1283...,5760,,
1,2013-01-02,/content/solar_data/data/raw/system_1283/year_...,/content/solar_data/data/processed/system_1283...,5760,,
2,2013-01-03,/content/solar_data/data/raw/system_1283/year_...,/content/solar_data/data/processed/system_1283...,5760,,
3,2013-01-04,/content/solar_data/data/raw/system_1283/year_...,/content/solar_data/data/processed/system_1283...,5760,,
4,2013-01-06,/content/solar_data/data/raw/system_1283/year_...,/content/solar_data/data/processed/system_1283...,5760,,


Raw code to re install all the files


In [36]:
from pathlib import Path
import shutil

# Use the project location already used in your notebook
PROJECT = Path("/content/solar_data")

# Create the required folders; existing folders remain unchanged
folders = [
    PROJECT / "data/raw/system_1283/year_2013",
    PROJECT / "data/raw/system_1283/year_2014",
    PROJECT / "data/processed",
    PROJECT / "scripts",
    PROJECT / "reports",
    PROJECT / "docs",
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
    print("Folder ready:", folder)

# Copy the existing quality reports into the reports folder
report_files = [
    Path("/content/solar_quality_report_2013_2014_checked.csv"),
    Path("/content/solar_missing_dates_2013_2014.csv"),
]

for source in report_files:
    destination = PROJECT / "reports" / source.name

    if not source.exists():
        print("Report not found:", source)
    elif destination.exists():
        print("Report already exists:", destination)
    else:
        shutil.copy2(source, destination)
        print("Report copied:", destination)

# Check whether the downloaded raw CSV files are still present
for year in [2013, 2014]:
    year_folder = PROJECT / f"data/raw/system_1283/year_{year}"
    csv_count = sum(1 for _ in year_folder.rglob("*.csv"))
    print(f"{year}: {csv_count} raw CSV files found")

Folder ready: /content/solar_data/data/raw/system_1283/year_2013
Folder ready: /content/solar_data/data/raw/system_1283/year_2014
Folder ready: /content/solar_data/data/processed
Folder ready: /content/solar_data/scripts
Folder ready: /content/solar_data/reports
Folder ready: /content/solar_data/docs
Report copied: /content/solar_data/reports/solar_quality_report_2013_2014_checked.csv
Report copied: /content/solar_data/reports/solar_missing_dates_2013_2014.csv
2013: 343 raw CSV files found
2014: 382 raw CSV files found


In [37]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime, timezone
from google.colab import files

content = Path("/content")
project = content / "solar_data"

if not project.is_dir():
    raise FileNotFoundError("Project folder /content/solar_data is missing.")

# Include everything saved inside the project folder
submission_files = {
    file for file in project.rglob("*") if file.is_file()
}

# Include reports and logs saved directly in /content
# Existing ZIP backups are excluded to avoid duplicating data.
for pattern in [
    "solar*.csv",
    "solar*.json",
    "*1283*.csv",
    "*1283*.json",
]:
    submission_files.update(
        file for file in content.glob(pattern) if file.is_file()
    )

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
zip_path = content / f"Solar_Project_Submission_{timestamp}.zip"

print(f"Packaging {len(submission_files)} files...")

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for file in sorted(submission_files):
        archive.write(
            file,
            arcname=str(file.relative_to(content))
        )

print("ZIP ready:", zip_path.name)
print(f"Size: {zip_path.stat().st_size / (1024**3):.2f} GB")

files.download(str(zip_path))

Packaging 1803 files...
ZIP ready: Solar_Project_Submission_20261010_182254_320457.zip
Size: 0.43 GB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>